# Lab 6: Word Sense Disambiguation

Disambiguate the words of the *SMTeuroparl* sentence pairs (test set) with WordNet, using two WSD algorithms (Lesk from NLTK and UKB from TextServer), compute the similarity of each pair using senses and the Jaccard distance, and compare it with the words (Lab 2) and lemmas (Lab 3) approaches using the Pearson correlation with the gold standard.

For words, lemmas and Lesk, all the preprocessing (tokens, POS tags and lemmas) is done with spaCy, so they all come from the same tokens. UKB is run by TextServer, which uses its own preprocessing (FreeLing).

## Setup

Import the libraries, download WordNet and the spaCy model, and import the TextServer client (`textserver.py` in `common/`).

In [1]:
import sys
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
if "google.colab" in sys.modules:
    %pip install -q nltk spacy pandas scipy
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

DATA_DIR = ROOT / "test_gold"
NLTK_DIR = ROOT / "nltk_data"
COMMON_DIR = ROOT / "common"
UKB_CACHE = COMMON_DIR / "ukb_senses.json"

# Notebook libs
import json
from collections import Counter
from getpass import getpass

import nltk
import pandas as pd
import requests
import spacy
from nltk.corpus import wordnet as wn
from nltk.metrics.distance import jaccard_distance
from nltk.wsd import lesk
from scipy.stats import pearsonr

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("wordnet", download_dir=NLTK_DIR)
nltk.download("omw-1.4", download_dir=NLTK_DIR)

# TextServer client (textserver.py in common/), used for UKB
sys.path.insert(0, str(COMMON_DIR))
from textserver import TextServer

# TextServer with the user and password of credentials.json in common/
# or asked for if it is not there.
def textserver_login(service):
    credentials_file = COMMON_DIR / "credentials.json"
    if credentials_file.exists():
        credentials = json.loads(credentials_file.read_text())
    else:
        credentials = {"user": input("TextServer user: "), "password": getpass("TextServer password: ")}
    return TextServer(credentials["user"], credentials["password"], service)

# SpaCy model needed by this notebook
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Data and preprocessing

Load the *SMTeuroparl* sentence pairs with their gold-standard scores, and process every sentence with spaCy.

**Why spaCy.** WSD needs the POS tag and the lemma of each word: the POS tag chooses the WordNet category (noun, verb, adjective or adverb) and the lemma is the word looked up in WordNet. spaCy gives the tokens, POS tags and lemmas in one pass, and in Lab 3 its lemmas were better than the NLTK ones (POS tagger + WordNet lemmatizer). 

**Case.** The spaCy lemmas are lowercased, except for proper nouns and *I* (*The* → *the* in the example below). They are used as given by spaCy, without extra lowercasing, so names keep their capital letter. WordNet ignores the case when it looks up a word, so the senses do not depend on it (checked in the following cell).

In [2]:
# Load the spaCy model (tokens, POS tags and lemmas).
nlp = spacy.load("en_core_web_sm")

# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / "STS.input.SMTeuroparl.txt", sep="\t", header=None)
dt["gs"] = pd.read_csv(DATA_DIR / "STS.gs.SMTeuroparl.txt", sep="\t", header=None)

# Process every sentence once with spaCy.
docs1 = [nlp(s) for s in dt[0]]
docs2 = [nlp(s) for s in dt[1]]
print(f"Sentence pairs: {len(dt)}")

# WordNet lowercases the word before looking it up, so the case does not change the senses.
print("\nSame synsets for 'Report' and 'report':", wn.synsets("Report", wn.NOUN) == wn.synsets("report", wn.NOUN))

Sentence pairs: 459



Same synsets for 'Report' and 'report': True


## Lesk (NLTK)

Disambiguate every noun, verb, adjective and adverb with `nltk.wsd.lesk`, using the spaCy POS tag as the WordNet category and the whole sentence as context.

In [3]:
# spaCy POS tags of the words that can have a WordNet sense.
# The other tags (determiners, prepositions, punctuation...) have no WordNet category.
wn_pos = {"NOUN": wn.NOUN, "PROPN": wn.NOUN, "VERB": wn.VERB, "ADJ": wn.ADJ, "ADV": wn.ADV}

# For every sentence, a list of (token, synset) pairs chosen by Lesk.
# The synset is None when the word has no sense.
senses1, senses2 = [], []

for docs, senses in [(docs1, senses1), (docs2, senses2)]:
    for doc in docs:
        # Lesk uses the whole sentence as context.
        context = [t.text for t in doc]
        sentence = []
        for token in doc:
            pos = wn_pos.get(token.pos_)
            # Look up the lemma, or the word itself if the lemma is not in WordNet.
            # The senses are passed with synsets=, because lesk(context, word, pos) drops the satellite adjectives (POS 's').
            word = token.lemma_ if pos and wn.synsets(token.lemma_, pos) else token.text
            synset = lesk(context, word, synsets=wn.synsets(word, pos)) if pos and wn.synsets(word, pos) else None
            sentence.append((token, synset))
        senses.append(sentence)

# Example: the senses of the first sentence.
print(docs1[0])
rows = []
for token, synset in senses1[0]:
    if synset:
        rows.append({"word": token.text, "lemma": token.lemma_, "POS": token.pos_,"sense": synset.name(), "definition": 
                     synset.definition()})
pd.DataFrame(rows).set_index("word")

The leaders have now been given a new chance and let us hope they seize it.


,lemma,POS,sense,definition
word,,,,
leaders,leader,NOUN,leader.n.01,a person who rules or guides or inspires others
now,now,ADV,now.r.01,in the historical present; at this point in th...
given,give,VERB,give.v.05,"convey, as of a compliment, regards, attention..."
new,new,ADJ,fresh.s.03,original and of a kind not seen before
chance,chance,NOUN,probability.n.01,a measure of how likely it is that some event ...
let,let,VERB,let.v.01,make it possible through a specific action or ...
hope,hope,VERB,hope.v.01,expect and wish
seize,seize,VERB,assume.v.06,seize and take control without authority and p...


## UKB (TextServer)

UKB is a graph-based WSD algorithm. It is run by the TextServer `senses` service, which also does its own tokenization, POS tagging and lemmatization (FreeLing).

Each distinct sentence is sent once to TextServer. The answers are saved in `common/ukb_senses.json`, so the server is only queried the first time the notebook is run. With this file, it runs without TextServer, and Lab 7 reuses it. The synsets are returned as WordNet 3.0 offsets (`10285313-n`), which are converted to NLTK synsets.

In [4]:
# Query TextServer once for every distinct sentence, and keep the answers in a JSON file.
sentences = sorted(set(dt[0]) | set(dt[1]))
ukb_raw = json.loads(UKB_CACHE.read_text()) if UKB_CACHE.exists() else {}
missing = [s for s in sentences if s not in ukb_raw]
if missing:
    ts = textserver_login("senses")
    for sentence in missing:
        # The server sometimes fails or does not answer, so try up to 3 times.
        for attempt in range(3):
            try:
                answer = ts.senses(sentence)
                break
            except requests.RequestException:
                if attempt == 2:
                    raise
        # TextServer may split a sentence in two, so join the tokens of all its sentences.
        ukb_raw[sentence] = [token for sent in answer for token in sent]
        # Save after every query, so an interrupted run does not lose the answers.
        UKB_CACHE.write_text(json.dumps(ukb_raw, ensure_ascii=False, indent=1))
print(f"Distinct sentences: {len(sentences)}   queried now: {len(missing)}")

# '10285313-n' -> Synset('male_child.n.01'), or None for 'N/A'.
def to_synset(wn_id):
    if wn_id == "N/A":
        return None
    offset, pos = wn_id.split("-")
    return wn.synset_from_pos_and_offset(pos, int(offset))

# For every sentence, a list of (lemma, synset) pairs chosen by UKB (None when the word has no sense).
ukb1 = [[(lemma, to_synset(wn_id)) for form, lemma, tag, pos, wn_id in ukb_raw[s]] for s in dt[0]]
ukb2 = [[(lemma, to_synset(wn_id)) for form, lemma, tag, pos, wn_id in ukb_raw[s]] for s in dt[1]]

# Example: the senses of the first sentence, with the Lesk sense of the same lemma.
print(dt[0][0])
lesk_by_lemma = {t.lemma_.lower(): ss for t, ss in senses1[0] if ss}
rows = []
for (form, lemma, tag, pos, wn_id), (_, synset) in zip(ukb_raw[dt[0][0]], ukb1[0]):
    if synset:
        rows.append({"word": form, "lemma": lemma, "POS": pos, "UKB sense": synset.name(),
                     "Lesk sense": lesk_by_lemma[lemma].name() if lemma in lesk_by_lemma else None,
                     "UKB definition": synset.definition()})
pd.DataFrame(rows).set_index("word")

Distinct sentences: 338   queried now: 0
The leaders have now been given a new chance and let us hope they seize it.


,lemma,POS,UKB sense,Lesk sense,UKB definition
word,,,,,
leaders,leader,noun,leader.n.01,leader.n.01,a person who rules or guides or inspires others
have,have,verb,get.v.03,NaN,cause to move; cause to be in a certain positi...
now,now,adverb,immediately.r.01,now.r.01,without delay or hesitation; with no time inte...
been,be,verb,be.v.01,NaN,"have the quality of being; (copula, used with ..."
given,give,verb,hold.v.03,give.v.05,organize or be responsible for
new,new,adjective,new.a.06,fresh.s.03,unaffected by use or exposure
chance,chance,noun,opportunity.n.01,probability.n.01,a possibility due to a favorable combination o...
let,let,verb,get.v.03,let.v.01,cause to move; cause to be in a certain positi...
hope,hope,verb,hope.v.02,hope.v.01,be optimistic; be full of hope; have hopes


## Similarities and Pearson correlation

Jaccard similarity of each pair using words (spaCy tokens), lemmas and senses (Lesk and UKB), and the Pearson correlation of each approach with the gold standard.

UKB uses the FreeLing lemmas of TextServer, which are all lowercased and differ a little from the spaCy ones. So the lemmas are also computed with the FreeLing lemmas, to compare UKB with lemmas from the same preprocessing.

In [5]:
def jaccard(set1, set2):
    # Two empty sets share nothing (with UKB, pair 260 has no sense in either sentence).
    if not set1 and not set2:
        return 0.0
    return 1 - jaccard_distance(set1, set2)

# Words (Lab 2) and lemmas (Lab 3), computed again with the spaCy tokens for the comparison.
dt["words"] = [jaccard({t.text for t in d1}, {t.text for t in d2}) for d1, d2 in zip(docs1, docs2)]
dt["lemmas"] = [jaccard({t.lemma_ for t in d1}, {t.lemma_ for t in d2}) for d1, d2 in zip(docs1, docs2)]

# FreeLing lemmas (TextServer), the same preprocessing as UKB.
dt["lemmas (FreeLing)"] = [jaccard({l for l, _ in x1}, {l for l, _ in x2}) for x1, x2 in zip(ukb1, ukb2)]

# Two sentence representations built from the senses:
# - "senses": only the synsets of the disambiguated words.
# - "senses + rest": the synsets, plus the lemma of the words without a sense
#   (function words, punctuation, numbers, words not in WordNet).
# Lesk gives (token, synset) pairs and UKB gives (lemma, synset) pairs.
def sense_set(disambiguated, keep_rest):
    items = set()
    for word, synset in disambiguated:
        if synset:
            items.add(synset.name())
        elif keep_rest:
            if isinstance(word, str):
                items.add(word)
            else:
                items.add(word.lemma_)
    return items

for name, x1s, x2s in [("Lesk", senses1, senses2), ("UKB", ukb1, ukb2)]:
    dt[f"{name} senses"] = [jaccard(sense_set(x1, False), sense_set(x2, False)) for x1, x2 in zip(x1s, x2s)]
    dt[f"{name} senses + rest"] = [jaccard(sense_set(x1, True), sense_set(x2, True)) for x1, x2 in zip(x1s, x2s)]

approaches = ["words", "lemmas", "lemmas (FreeLing)", "Lesk senses", "Lesk senses + rest", "UKB senses", "UKB senses + rest"]
summary = pd.DataFrame({"Pearson": [pearsonr(dt["gs"], dt[a])[0] for a in approaches]}, index=approaches)
summary.round(4)

,Pearson
words,0.4606
lemmas,0.4784
lemmas (FreeLing),0.4433
Lesk senses,0.4406
Lesk senses + rest,0.4288
UKB senses,0.4164
UKB senses + rest,0.4145


## Senses vs. lemmas, pair by pair

As in Lab 3, count the pairs whose similarity increases or decreases with senses (compared with lemmas), and look at the pairs with the largest changes with each algorithm to see why.

The comparison uses "senses + rest", against the lemmas of the same preprocessing (spaCy for Lesk, FreeLing for UKB): it keeps the lemma of the words without a sense, so it only differs from lemmas in the words that get a sense, and every change comes from the disambiguation. The effect of leaving out the words without a sense is already given by "senses" vs. "senses + rest" in the table above.

In [6]:
def show_pair(i, wsd):
    print(f"Pair {i}  (gold {dt['gs'][i]})")
    print(f"  S1: {dt[0][i]}")
    print(f"  S2: {dt[1][i]}")
    print("  " + "   ".join(f"{a} {dt[a][i]:.3f}" for a in approaches))
    if wsd == "Lesk":
        representations = [("Lemmas", {t.lemma_ for t in docs1[i]}, {t.lemma_ for t in docs2[i]}),
                           ("Lesk senses + rest", sense_set(senses1[i], True), sense_set(senses2[i], True))]
    else:
        representations = [("Lemmas (FreeLing)", {l for l, _ in ukb1[i]}, {l for l, _ in ukb2[i]}),
                           ("UKB senses + rest", sense_set(ukb1[i], True), sense_set(ukb2[i], True))]
    for name, set1, set2 in representations:
        print(f"  {name}: shared {sorted(set1 & set2)}")
        print(f"  {'':<{len(name)}}  only S1 {sorted(set1 - set2)}")
        print(f"  {'':<{len(name)}}  only S2 {sorted(set2 - set1)}")
    print()

# Pairs whose similarity increases / decreases / stays equal with senses, compared with lemmas.
dt["Lesk diff"] = dt["Lesk senses + rest"] - dt["lemmas"]
dt["UKB diff"] = dt["UKB senses + rest"] - dt["lemmas (FreeLing)"]
for name in ["Lesk", "UKB"]:
    diff = dt[f"{name} diff"]
    print(f"{name} senses + rest vs. lemmas: higher {(diff > 0).sum()}, lower {(diff < 0).sum()}, equal {(diff == 0).sum()}")

Lesk senses + rest vs. lemmas: higher 36, lower 200, equal 223
UKB senses + rest vs. lemmas: higher 88, lower 107, equal 264


### Largest increases and decreases (Lesk)

In [7]:
# The 3 largest increases and decreases of the given algorithm compared with its lemmas.
def show_largest_changes(wsd):
    # Many pairs share the same first sentence (Lab 3), so keep only the largest change of each one.
    increases = dt.sort_values(f"{wsd} diff", ascending=False, kind="stable").drop_duplicates(subset=[0])
    decreases = dt.sort_values(f"{wsd} diff", kind="stable").drop_duplicates(subset=[0])
    print("=== Largest increases ===\n")
    for i in increases.index[:3]:
        show_pair(i, wsd)
    print("=== Largest decreases ===\n")
    for i in decreases.index[:3]:
        show_pair(i, wsd)

show_largest_changes("Lesk")

=== Largest increases ===

Pair 260  (gold 4.75)
  S1: Van Orden Report (A5-0241/2000)
  S2: Report Van Orden (A5-0241/2000)
  words 1.000   lemmas 0.778   lemmas (FreeLing) 0.600   Lesk senses 1.000   Lesk senses + rest 1.000   UKB senses 0.000   UKB senses + rest 0.600
  Lemmas: shared ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'Van']
          only S1 ['Report']
          only S2 ['report']
  Lesk senses + rest: shared ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'avant-garde.n.01', 'report.n.01']
                      only S1 []
                      only S2 []

Pair 356  (gold 4.0)
  S1: Then perhaps we could have avoided a catastrophe.
  S2: We would maybe then have been able to avoid a disaster.
  words 0.167   lemmas 0.400   lemmas (FreeLing) 0.400   Lesk senses 0.800   Lesk senses + rest 0.615   UKB senses 0.714   UKB senses + rest 0.615
  Lemmas: shared ['.', 'a', 'avoid', 'have', 'then', 'we']
          only S1 ['catastrophe', 'could', 'perhaps']
          only S2 ['able', '

### Largest increases and decreases (UKB)

In [8]:
show_largest_changes("UKB")

=== Largest increases ===

Pair 356  (gold 4.0)
  S1: Then perhaps we could have avoided a catastrophe.
  S2: We would maybe then have been able to avoid a disaster.
  words 0.167   lemmas 0.400   lemmas (FreeLing) 0.400   Lesk senses 0.800   Lesk senses + rest 0.615   UKB senses 0.714   UKB senses + rest 0.615
  Lemmas (FreeLing): shared ['.', 'a', 'avoid', 'have', 'then', 'we']
                     only S1 ['can', 'catastrophe', 'perhaps']
                     only S2 ['able', 'be', 'disaster', 'maybe', 'to', 'would']
  UKB senses + rest: shared ['.', 'a', 'avoid.v.01', 'calamity.n.01', 'possibly.r.01', 'suffer.v.02', 'then.r.02', 'we']
                     only S1 ['can']
                     only S2 ['able.a.01', 'be.v.01', 'to', 'would']

Pair 42  (gold 5.0)
  S1: Amendment No 7 proposes certain changes in the references to paragraphs.
  S2: Amendment 7 proposes certain modifications in the references to the paragraphs.
  words 0.769   lemmas 0.769   lemmas (FreeLing) 0.769   Lesk

## Same lemma, different sense

Both algorithms disambiguate each sentence on its own, so a word that appears in both sentences of a pair can get a different sense in each one, although it has the same meaning (*adopt* and *resolution* in pair 12, *aid*, *market*, *grass* and *essential* in pair 187, shown above). List the lemmas for which this happens most often with Lesk, and compare the senses chosen by Lesk and UKB for the most frequent one.

In [9]:
# lemma -> sense of the words of a sentence that got a sense.
def lemma_senses(x):
    result = {}
    for word, synset in x:
        if synset is None:
            continue
        if isinstance(word, str):
            lemma = word
        else:
            lemma = word.lemma_
        result[lemma.lower()] = synset
    return result

# Lemmas that most often get a different sense with Lesk (on the spaCy lemmas) in the two sentences of a pair.
mismatches = Counter()
examples = {}
for i, (x1, x2) in enumerate(zip(senses1, senses2)):
    s1, s2 = lemma_senses(x1), lemma_senses(x2)
    for l in s1.keys() & s2.keys():
        if s1[l] != s2[l]:
            mismatches[l] += 1
            examples.setdefault(l, (i, s1[l], s2[l]))

print("Lesk (spaCy): lemmas with a different sense in the two sentences of a pair")
for lemma, count in mismatches.most_common(6):
    i, ss1, ss2 = examples[lemma]
    print(f"{lemma:<12} {count:>3} pairs   e.g. pair {i}: {ss1.name()} / {ss2.name()}")

Lesk (spaCy): lemmas with a different sense in the two sentences of a pair
report        28 pairs   e.g. pair 18: report.n.02 / report.n.01
reading       17 pairs   e.g. pair 17: reading.n.03 / reading.n.01
grass         17 pairs   e.g. pair 28: grass.n.01 / eatage.n.01
act           17 pairs   e.g. pair 65: work.v.03 / act.v.02
union         15 pairs   e.g. pair 79: union.n.01 / coupling.n.03
amendment     14 pairs   e.g. pair 1: amendment.n.01 / amendment.n.02


In [10]:
# The senses chosen by Lesk and UKB for "report", the lemma with most different senses, with their definitions.
i = examples["report"][0]
for sentence, x, u in [(dt[0][i], senses1[i], ukb1[i]), (dt[1][i], senses2[i], ukb2[i])]:
    print(sentence)
    for token, ss in x:
        if ss and token.lemma_.lower() == "report":
            print(f"  Lesk {token.text} -> {ss.name()}: {ss.definition()}")
    for l, ss in u:
        if ss and l == "report":
            print(f"  UKB  {l} -> {ss.name()}: {ss.definition()}")

Mr President, the Cashman report can be summarised in four words: citizens' power over bureaucracy.
  Lesk report -> report.n.02: the act of informing by verbal report
  UKB  report -> report.n.03: a short account of the news
Mr President, the Cashman report can be summed up in a few words: the power of citizens on bureaucracy.
  Lesk report -> report.n.01: a written document describing the findings of some individual or group
  UKB  report -> report.n.03: a short account of the news


## Synonyms merged by senses

The largest increases above come from different lemmas that get the same synset (pairs 356, 7, 42, 219). List the lemmas that are merged most often by each algorithm.

In [11]:
# Senses also merge different words with the same meaning (synonyms share the synset).
# Count the pairs where two different lemmas end in the same synset.
for name, x1s, x2s in [("Lesk", senses1, senses2), ("UKB", ukb1, ukb2)]:
    merged = Counter()
    for x1, x2 in zip(x1s, x2s):
        by_synset1 = {ss: l for l, ss in lemma_senses(x1).items()}
        by_synset2 = {ss: l for l, ss in lemma_senses(x2).items()}
        for ss in by_synset1.keys() & by_synset2.keys():
            if by_synset1[ss] != by_synset2[ss]:
                merged[(by_synset1[ss], by_synset2[ss], ss.name())] += 1

    print(f"{name}: different lemmas mapped to the same synset in {sum(merged.values())} cases")
    for (l1, l2, ss_name), count in merged.most_common(8):
        print(f"  {l1} / {l2} -> {ss_name} ({count} pairs)")
    print()

Lesk: different lemmas mapped to the same synset in 69 cases
  catastrophe / disaster -> catastrophe.n.02 (14 pairs)
  hence / therefore -> therefore.r.01 (14 pairs)
  scarcely / hardly -> barely.r.01 (13 pairs)
  summarise / sum -> summarize.v.02 (7 pairs)
  summarise / summarize -> summarize.v.02 (5 pairs)
  mr / mister -> mister.n.01 (4 pairs)
  mr / mr. -> mister.n.01 (2 pairs)
  perhaps / maybe -> possibly.r.01 (2 pairs)

UKB: different lemmas mapped to the same synset in 91 cases
  catastrophe / disaster -> calamity.n.01 (14 pairs)
  big / large -> large.a.01 (14 pairs)
  hence / therefore -> therefore.r.01 (14 pairs)
  scarcely / hardly -> hardly.r.02 (13 pairs)
  quite / rather -> quite.r.01 (9 pairs)
  change / modification -> change.n.01 (5 pairs)
  summarise / summarize -> sum_up.v.01 (5 pairs)
  fervent / ardent -> ardent.s.01 (5 pairs)



## Questions

### Approach

Two WSD algorithms are used, both with WordNet 3.0:

- **Lesk** (`nltk.wsd.lesk`) chooses the synset whose definition shares the most words with the sentence. Each sentence is processed with spaCy (tokens, POS tags and lemmas), which is also used for words and lemmas. The POS tag gives the WordNet category of every noun, verb, adjective and adverb, and the context is the whole sentence.
- **UKB** (TextServer, service `senses`) runs Personalized PageRank over the WordNet graph, starting from the senses of all the words of the sentence. TextServer does its own preprocessing (FreeLing), so the FreeLing lemmas are also evaluated, to compare UKB with lemmas from the same tokens.

The sentence is then represented as the set of its synsets ("senses"), or as the synsets plus the lemmas of the words without a sense ("senses + rest"), and the two sets are compared with the Jaccard coefficient.

### Comparison with words (Lab 2) and lemmas (Lab 3)

Words, lemmas and Lesk start from the same tokens and only the representation changes.

| Approach | Pearson |
|---|---|
| Words | 0.4606 |
| **Lemmas** | **0.4784** |
| Lemmas (FreeLing) | 0.4433 |
| Lesk senses | 0.4406 |
| Lesk senses + rest | 0.4288 |
| UKB senses | 0.4164 |
| UKB senses + rest | 0.4145 |

- **Senses give a lower correlation than lemmas with both algorithms.** Lesk senses (0.4406) are below the spaCy lemmas (0.4784) and words (0.4606), and UKB senses (0.4164) are below the FreeLing lemmas of the same preprocessing (0.4433).
- UKB gets the lowest correlation of the table, but part of it comes from the preprocessing and not from the disambiguation: the FreeLing lemmas alone are already 0.035 below the spaCy lemmas. Compared with the lemmas of its own preprocessing, UKB loses less (0.4433 → 0.4145) than Lesk (0.4784 → 0.4288).
- Adding the words without a sense lowers the correlation a little with both algorithms (Lesk 0.4406 → 0.4288, UKB 0.4164 → 0.4145). These are mostly function words and punctuation, which are shared by most pairs and do not carry the meaning.

### Senses vs. lemmas, pair by pair

To see the effect of the disambiguation alone, "senses + rest" is compared with the lemmas of the same preprocessing: the two sets only differ in the words that get a sense. **With Lesk, the disambiguation lowers the similarity of many more pairs than it raises**: higher for 36 pairs, lower for 200 and equal for 223. With UKB the effect is much more balanced: higher for 88, lower for 107 and equal for 264.

The largest changes with Lesk show where the senses help and where they hurt.

The largest increases come from **different lemmas that get the same synset**:

- Pair 260 (gold 4.75): *Report* / *report* are different lemmas (the capital letter is kept), but WordNet ignores the case and both get `report.n.01` (0.778 → 1.000).
- Pair 356 (gold 4.0): the synonyms *catastrophe* / *disaster* get `catastrophe.n.02` and *perhaps* / *maybe* get `possibly.r.01` (0.400 → 0.615).
- Pair 7 (gold 5.0): *fishermen* / *fishers* both get `fisherman.n.01` (0.800 → 1.000).

The largest decreases come from **the same lemma getting a different sense in each sentence**:

- Pair 12 (gold 4.75): the lemmas are identical (1.000), but *adopt* gets `espouse.v.03` / `adopt.v.01` and *resolution* gets `resolving_power.n.01` / `resolution.n.01` (→ 0.556). UKB gives the same senses in both sentences and keeps 1.000.
- Pair 209 (gold 4.25): *whole* gets `whole.n.02` / `whole.n.01`, and *balance* is a noun in S1 and a verb (*balanced*) in S2, so it gets senses of different categories (0.556 → 0.273).
- Pair 187 (gold 4.25): *aid*, *market*, *grass* and *essential* get a different sense in each sentence, for example `grass.n.01` / `eatage.n.01` (0.412 → 0.143). With UKB this pair stays at 0.412.

With UKB, the largest changes (compared with the FreeLing lemmas) have the same two causes:

- Increases from synonyms: in pair 42 *change* / *modification* both get `change.n.01` (0.769 → 0.917), in pair 219 *whole* / *unit* both get `whole.n.02` (0.364 → 0.500), and pair 356 appears again (*catastrophe* / *disaster* → `calamity.n.01`).
- Decreases from the same lemma getting a different sense: in pair 111 *place* gets `topographic_point.n.01` / `position.n.06` (0.889 → 0.700), in pair 209 *balance* is again a noun and a verb (`balance.n.04` / `balance.v.01`, 0.556 → 0.400), and in pair 37 *qualify* gets `qualify.v.04` / `modify.v.02` (0.500 → 0.364). In pair 37 UKB also gives *House* (the Parliament) the sense `theater.n.01`.

### Why the senses do not help

The sentences are short, so the algorithms have very little context. Lesk chooses the sense whose definition shares the most words with the sentence, so a small change in the wording changes the chosen sense: **the same word gets a different sense in each sentence of a pair, although it has the same meaning** (pairs 12, 209 and 187). *report* gets different senses in 28 pairs, for example `report.n.02` / `report.n.01` in pair 18.

UKB uses the whole WordNet graph and is **more consistent**: in pair 18 *report* gets `report.n.03` in both sentences, and pairs 12 and 187 keep the similarity of the lemmas. But consistent is not correct: `report.n.03` is "a short account of the news".

The only gain of senses is that **synonyms get the same synset** (*catastrophe* / *disaster*, *hence* / *therefore*, 14 pairs each with both algorithms), and it is smaller than the loss.

### Conclusion

Neither algorithm gives senses that are better than lemmas: Lesk senses + rest (0.4288) are below the spaCy lemmas (0.4784), and UKB senses + rest (0.4145) are below the FreeLing lemmas (0.4433). Senses could only help if the disambiguation were correct and consistent between the two sentences, which is hard with such short sentences.